# Exploração · `bronze.unidades_consumidoras`

Notebook de **investigação**, fora do pipeline. Descobre os problemas de qualidade do cadastro de UCs e a regra de limpeza de cada um, antes de escrever a Silver.

**Status: concluída.** A Silver está em `02_silver_unidades_consumidoras.ipynb`. Rode com **Run all**: cada célula só depende das anteriores. Os achados são registrados nas células markdown e, depois, em `docs/data_quality_log.md` (seção `unidades_consumidoras`, códigos U).

É a **tabela mais central do case**: todas as outras se ligam a ela por `id_uc`, e a Gold vai ter uma `dim_uc` com os dados daqui.

| # | Seção | Pergunta |
|---|---|---|
| 1 | Visão geral | Colunas, tipos e volume |
| 2 | Cabeçalho repetido | Confirmar U1 (1 linha) |
| 3 | Nulos | Quais colunas têm vazios? |
| 4 | Duplicatas | As 1.260 linhas repetidas são cópias exatas? Se não, o que muda entre elas? |
| 5 | Formatos | `id_uc`, `cpf_cnpj`, `cep`, `numero`, latitude e longitude |
| 6 | Categorias | `uf`, `municipio`, `bairro`, `classe`, `subgrupo_tarifario`, `status` |
| 7 | Classe × subgrupo | A classe bate com o subgrupo tarifário? |
| 8 | Alimentador × município | O código do alimentador bate com o município? |
| 9 | `data_ligacao` | Formatos, ordem dia/mês e datas impossíveis |
| 10 | As 200 UCs sem medidor | Quem são? Por que não têm leitura? |
| 11 | Dados pessoais (LGPD) | O que precisa de máscara na Gold |
| 12 | Resumo | Regras propostas para a Silver |

**Já sabemos:**
- 46.461 linhas na Bronze, sendo 1 cabeçalho repetido no arquivo de origem (U1).
- Só 45.200 `id_uc` distintos: **1.260 linhas repetidas** (exploração de medidores, seção 9).
- 200 UCs `ATIVA` sem medidor ativo e sem nenhuma leitura (exploração de medidores, seção 9).
- Achados da inspeção: `uf` com `PE` e `Pernambuco`, logradouro com espaço no fim, e dados pessoais (`cpf_cnpj`, `nome_titular`).

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

catalog = "v_energy"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

ucs_raw = spark.table(f"{bronze_schema}.unidades_consumidoras")

def n_br(n):
    return f"{n:,}".replace(",", ".")

def sem_acento(col):
    """upper + trim + sem acento + espaços internos simples: para comparar grafias."""
    c = F.upper(F.trim(col))
    c = F.translate(c, "ÁÀÂÃÉÊÍÓÔÕÚÜÇ", "AAAAEEIOOOUUC")
    return F.regexp_replace(c, r"\s+", " ")

## 1. Visão geral

**Achado:** 46.461 linhas, todas as colunas como texto (padrão da Bronze), mais as colunas de auditoria.

In [0]:
ucs_raw.printSchema()
display(ucs_raw.limit(5))
print(f"{n_br(ucs_raw.count())} linhas")

## 2. Cabeçalho repetido (U1)

**O que explora:** confirma a linha em que cada coluna tem o próprio nome. A partir daqui, as análises usam `ucs` (sem essa linha).

**Achado:** confirmado: 1 linha de cabeçalho repetido (U1). Sem ela, 46.460 linhas.

In [0]:
cab = ucs_raw.filter(F.col("id_uc") == "id_uc")
print("linhas de cabeçalho:", cab.count())  # esperado: 1
display(cab)

ucs = ucs_raw.filter(~F.coalesce(F.col("id_uc") == "id_uc", F.lit(False)))
cols_origem = [c for c in ucs.columns if c not in ("source_file", "ingestion_timestamp")]
print("linhas sem o cabeçalho:", n_br(ucs.count()))

## 3. Nulos

**O que explora:** nulos reais e textos vazios por coluna.

**Achado:** as colunas obrigatórias vêm preenchidas; os problemas do cadastro são de forma (grafias, máscaras, codificação), não de ausência. O `cep` é a exceção: tem valores que não são CEP (seção 5).

In [0]:
display(ucs.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols_origem]))
display(ucs.select([F.sum((F.trim(F.col(c)) == "").cast("int")).alias(c) for c in cols_origem]))

## 4. Duplicatas

**O que explora:** se as repetições de `id_uc` são cópias exatas (linhas distintas = ids distintos) ou se a mesma UC aparece com dados diferentes. Nesse caso, mostra **quais colunas mudam** entre as versões. Isso decide a regra da Silver: `dropDuplicates` simples (cópias) ou uma regra para escolher a versão certa.

**Achado:** 46.460 linhas para 45.200 `id_uc` distintos, mas as repetições **não são cópias exatas**: a mesma UC aparece com `nome_titular` e `cpf_cnpj` escritos de jeitos diferentes (máscara, acento, maiúsculas). Depois de padronizar essas duas colunas, **0 ids continuam com versões diferentes** (seção 13). Regra: padronizar e depois `dropDuplicates(["id_uc"])` (U2).

In [0]:
print("linhas:                          ", n_br(ucs.count()))
print("id_uc distintos:                 ", n_br(ucs.select("id_uc").distinct().count()))
print("linhas distintas (sem auditoria):", n_br(ucs.select(cols_origem).distinct().count()))

# quantas vezes cada id aparece
display(ucs.groupBy("id_uc").count().filter("count > 1")
        .groupBy("count").agg(F.count("*").alias("qtd_ids")).orderBy("count"))

In [0]:
# Ids repetidos com conteúdo diferente: quais colunas mudam entre as versões?
versoes = ucs.select(cols_origem).distinct()
ids_diferentes = versoes.groupBy("id_uc").count().filter("count > 1").select("id_uc")
print("ids com versões diferentes:", n_br(ids_diferentes.count()))

dif = versoes.join(ids_diferentes, "id_uc")
display(dif.groupBy("id_uc").agg(
    *[(F.countDistinct(F.coalesce(F.col(c), F.lit("<nulo>"))) > 1).cast("int").alias(c)
      for c in cols_origem if c != "id_uc"]
).select([F.sum(c).alias(c) for c in cols_origem if c != "id_uc"]))

display(dif.orderBy("id_uc").limit(20))

In [0]:
# Se as diferenças forem só de grafia (maiúsculas, acento, espaço), elas somem depois de padronizar
texto = ["nome_titular", "logradouro", "bairro", "municipio", "uf", "classe", "status"]
padronizado = dif.select("id_uc", *[sem_acento(F.col(c)).alias(c) for c in texto],
                         *[c for c in cols_origem if c not in texto and c != "id_uc"]).distinct()
print("ids que continuam com versões diferentes depois de padronizar o texto:",
      n_br(padronizado.groupBy("id_uc").count().filter("count > 1").count()))

## 5. Formatos

**O que explora:**
- `id_uc` = `UC` + 6 dígitos (padrão das outras tabelas).
- `cpf_cnpj`: só dígitos? 11 (CPF) ou 14 (CNPJ)? Com máscara (`.`, `-`, `/`)? Zeros à esquerda perdidos (10 ou 13 dígitos)?
- `cep`: 8 dígitos, com ou sem hífen?
- `numero`: só número, ou textos como `S/N`?
- `latitude`/`longitude`: numéricas e dentro da região metropolitana do Recife (aprox. lat −7,5 a −8,6; lon −34,8 a −35,4).

**Achado:** `id_uc` sempre no padrão `UC` + 6 dígitos. `cpf_cnpj` vem com e sem máscara; só com dígitos, tem 11 (PF) ou 14 (PJ) dígitos (U8). `cep`: 1.383 linhas em 7 formas que não são CEP, que viram nulo (U7). Coordenadas numéricas.

In [0]:
pad = lambda c: F.regexp_replace(F.regexp_replace(F.col(c), "[0-9]", "9"), "[A-Za-z]", "A")

print("id_uc no padrão UC + 6 dígitos:")
display(ucs.groupBy(F.col("id_uc").rlike("^UC[0-9]{6}$").alias("no_padrao")).count())

for c in ["cpf_cnpj", "cep", "numero"]:
    print(c)
    display(ucs.groupBy(pad(c).alias("padrao")).count().orderBy(F.desc("count")).limit(15))

In [0]:
coords = (ucs
    .withColumn("lat", F.expr("try_cast(latitude as double)"))
    .withColumn("lon", F.expr("try_cast(longitude as double)")))

print("latitude que não vira número: ", n_br(coords.filter(F.col("latitude").isNotNull() & F.col("lat").isNull()).count()))
print("longitude que não vira número:", n_br(coords.filter(F.col("longitude").isNotNull() & F.col("lon").isNull()).count()))
display(coords.agg(F.min("lat"), F.max("lat"), F.min("lon"), F.max("lon")))
print("fora da região (lat −7,5 a −8,6; lon −34,8 a −35,4):",
      n_br(coords.filter(~F.col("lat").between(-8.6, -7.5) | ~F.col("lon").between(-35.4, -34.8)).count()))

## 6. Categorias

**O que explora:** valores distintos como vieram e depois de padronizar (`upper`, `trim`, sem acento). Se o número cai depois da padronização, havia grafias diferentes para o mesmo valor (ex.: `Jaboatão` e `JABOATAO`, `PE` e `Pernambuco`).

**Achado:** `uf` com 6.990 linhas fora de `PE` (`Pernambuco` e variações), todas viram `PE` (U4). `classe`, `municipio` e `status` têm grafias diferentes para o mesmo valor, que somem depois de padronizar (U5). No `bairro`, além das grafias, há **mojibake** (`JordÃ£o`, `BrasÃ­lia`), só nessa coluna (seção 13, U6). O mesmo nome de bairro pode existir em municípios diferentes, então a chave de bairro na Gold é **município + bairro**.

In [0]:
for c in ["uf", "municipio", "classe", "subgrupo_tarifario", "status"]:
    brutos = ucs.select(c).distinct().count()
    padronizados = ucs.select(sem_acento(F.col(c))).distinct().count()
    print(f"{c}: {brutos} valores brutos -> {padronizados} depois de padronizar")
    display(ucs.groupBy(c).count().orderBy(F.desc("count")))

In [0]:
# Bairro: muitos valores, então mostra só os que têm mais de uma grafia
b = ucs.select("bairro", "municipio").withColumn("bairro_pad", sem_acento(F.col("bairro")))
print("bairros brutos:", b.select("bairro").distinct().count(),
      "| depois de padronizar:", b.select("bairro_pad").distinct().count())
display(b.groupBy("bairro_pad").agg(F.countDistinct("bairro").alias("grafias"), F.collect_set("bairro").alias("exemplos"))
         .filter("grafias > 1").orderBy(F.desc("grafias")).limit(30))

# O mesmo bairro aparece em mais de um município? (pode ser legítimo, ex.: 'Centro')
display(b.groupBy("bairro_pad").agg(F.countDistinct(sem_acento(F.col("municipio"))).alias("municipios"))
         .filter("municipios > 1").orderBy(F.desc("municipios")).limit(20))

## 7. Classe × subgrupo tarifário

O glossário do case define: residencial = **B1**, rural = **B2**, comercial e poder público = **B3**, industrial = **A4** (média tensão). Uma combinação fora disso é erro de cadastro, e o subgrupo define a tarifa, então afeta o **preço** do diagnóstico.

**Achado:** todas as combinações seguem o glossário (RESIDENCIAL = B1, RURAL = B2, COMERCIAL e PODER PUBLICO = B3, INDUSTRIAL = A4). Vira checagem na Silver.

In [0]:
display(ucs.groupBy(sem_acento(F.col("classe")).alias("classe"), F.upper(F.trim("subgrupo_tarifario")).alias("subgrupo"))
        .count().orderBy("classe", F.desc("count")))

## 8. Alimentador × município

O código do alimentador parece trazer o município (`ALM-REC-004` = Recife, `ALM-JAB-001` = Jaboatão, `ALM-IGA-004` = Igarassu). Se o código não bater com o município da UC, há erro no cadastro, e isso afeta DEC/FEC por alimentador. Verifica também o padrão `ALM-XXX-000`.

**Achado:** todos os alimentadores seguem `ALM-XXX-000`, e cada código corresponde a um único município (14 códigos para os 14 municípios). Vira checagem na Silver.

In [0]:
al = ucs.withColumn("cod_mun_alim", F.regexp_extract("alimentador", r"^ALM-([A-Z]{3})-[0-9]{3}$", 1))

print("alimentador fora do padrão ALM-XXX-000:", n_br(al.filter(F.col("cod_mun_alim") == "").count()))
print("alimentadores distintos:", al.select("alimentador").distinct().count())

# Código do alimentador x município: cada código deveria ter um município só
display(al.groupBy("cod_mun_alim", sem_acento(F.col("municipio")).alias("municipio"))
          .count().orderBy("cod_mun_alim", F.desc("count")))

In [0]:
# UCs por alimentador: denominador do DEC/FEC (contar UCs distintas, não linhas)
por_alim = ucs.dropDuplicates(["id_uc"]).groupBy("alimentador").count()
display(por_alim.agg(F.count("*").alias("alimentadores"), F.min("count"), F.expr("percentile_approx(count, 0.5)"), F.max("count")))

## 9. `data_ligacao`

**O que explora:** formatos (mesma técnica das outras tabelas), datas impossíveis (futuras ou muito antigas) e a ordem dia/mês dos formatos não ISO. **Gabarito:** a `data_instalacao` do primeiro medidor da UC é igual à ligação em ~87% dos casos (exploração de medidores, seção 7.2). Agora o teste é ao contrário: usar a instalação (já limpa na `silver.medidores`) para confirmar a ligação.

**Achado:** a data de ligação é **igual** à instalação do primeiro medidor em todas as UCs, exceto em **90 com ligação antes de 1950**, quase todas em `01/01` (só 1 das 90 bate dia e mês com a instalação). São valores de preenchimento, não datas reais (seção 13, U9).

In [0]:
lig = ucs.withColumn("padrao", F.regexp_replace(F.coalesce("data_ligacao", F.lit("<nulo>")), "[0-9]", "9"))
display(lig.groupBy("padrao").count().orderBy(F.desc("count")))
print("sentinela 9999-12-31:", n_br(ucs.filter(F.col("data_ligacao") == "9999-12-31").count()))

nao_iso = (lig.filter(F.col("padrao").isin("99-99-9999", "99.99.9999", "99/99/9999"))
           .withColumn("p1", F.substring("data_ligacao", 1, 2).cast("int"))
           .withColumn("p2", F.substring("data_ligacao", 4, 2).cast("int")))
display(nao_iso.groupBy("padrao").agg(
    F.count("*").alias("linhas"),
    F.sum((F.col("p1") > 12).cast("int")).alias("primeira_parte_maior_12"),
    F.sum((F.col("p2") > 12).cast("int")).alias("segunda_parte_maior_12")))

In [0]:
# Gabarito: data de instalação do primeiro medidor da UC (silver.medidores já tem as datas convertidas)
def converte(col, fmt):
    return F.expr(f"to_date(try_to_timestamp({col}, '{fmt}'))")

prim_med = (spark.table(f"{silver_schema}.medidores")
            .groupBy("id_uc").agg(F.min("data_instalacao").alias("instalacao_1o_medidor")))

candidatos = {
    "99-99-9999": ["MM-dd-yyyy", "dd-MM-yyyy"],
    "99.99.9999": ["dd.MM.yyyy", "MM.dd.yyyy"],
    "99/99/9999": ["dd/MM/yyyy", "MM/dd/yyyy"],
}
iso = lig.filter(F.col("padrao") == "9999-99-99").withColumn("ligacao", converte("data_ligacao", "yyyy-MM-dd")).join(prim_med, "id_uc")
print("ISO: ligação = instalação do 1º medidor:", n_br(iso.filter("ligacao = instalacao_1o_medidor").count()), "de", n_br(iso.count()))

nao_iso_g = nao_iso.join(prim_med, "id_uc")
for padrao, fmts in candidatos.items():
    sub = nao_iso_g.filter(F.col("padrao") == padrao)
    for fmt in fmts:
        conv = sub.withColumn("ligacao", converte("data_ligacao", fmt))
        print(f"{padrao} como {fmt}: linhas {n_br(sub.count())} | não converte {n_br(conv.filter('ligacao IS NULL').count())}"
              f" | bate com a instalação {n_br(conv.filter('ligacao = instalacao_1o_medidor').count())}")

In [0]:
# Datas impossíveis (só ISO por enquanto)
d = lig.filter(F.col("padrao") == "9999-99-99").withColumn("ligacao", converte("data_ligacao", "yyyy-MM-dd"))
print("ligação depois de 2026-06-30:", n_br(d.filter("ligacao > DATE'2026-06-30'").count()))
display(d.groupBy(F.year("ligacao").alias("ano_ligacao")).count().orderBy("ano_ligacao"))

## 10. As 200 UCs ativas sem medidor e sem leitura

**O que explora:** quem são essas UCs. Hipóteses: (a) ligadas depois do fim das leituras, o que seria normal; (b) cópias "fantasmas" de outras UCs (mesmo endereço ou titular); (c) erro de cadastro. Compara a data de ligação, a classe, o município e o endereço com o resto do cadastro.

**Achado:** as 200 UCs sem medidor são **registros de teste** do sistema de origem, não clientes: não têm medidor, leitura nem fatura. Sem elas, o total é **45.000, o número do case**. Regra: remover (U3).

In [0]:
uc1 = ucs.dropDuplicates(["id_uc"])
com_medidor = spark.table(f"{silver_schema}.medidores").select("id_uc").distinct()
fantasmas = uc1.join(com_medidor, "id_uc", "left_anti")
print("UCs sem nenhum medidor:", n_br(fantasmas.count()))

display(fantasmas.limit(20))
display(fantasmas.groupBy(sem_acento(F.col("status")).alias("status"), sem_acento(F.col("classe")).alias("classe")).count())
display(fantasmas.groupBy(F.substring("data_ligacao", 1, 4).alias("ano_ligacao_texto")).count().orderBy("ano_ligacao_texto"))

In [0]:
# São duplicatas de outras UCs (mesmo titular ou mesmo endereço)?
outras = uc1.join(fantasmas.select("id_uc"), "id_uc", "left_anti")
print("fantasmas com o mesmo cpf_cnpj de outra UC:",
      n_br(fantasmas.join(outras.select("cpf_cnpj").distinct(), "cpf_cnpj", "left_semi").count()))
print("fantasmas com o mesmo endereço (logradouro + número + bairro) de outra UC:",
      n_br(fantasmas.join(outras.select("logradouro", "numero", "bairro").distinct(),
                          ["logradouro", "numero", "bairro"], "left_semi").count()))

## 11. Dados pessoais (LGPD)

**O que explora:** `cpf_cnpj` e `nome_titular` identificam pessoas. A Silver pode mantê-los (camada interna), mas a **Gold, o portal e o agente** só devem ver versões mascaradas (proposta no contrato da Gold). Aqui só se mede o que existe: quantos são PF (11 dígitos) e PJ (14 dígitos), e se o mesmo documento aparece em várias UCs (um cliente pode ter várias UCs, como diz o glossário).

**Achado:** `cpf_cnpj` e `nome_titular` são dados pessoais. Ficam na Silver (camada interna) e são **mascarados na Gold**, no portal e no agente (U10).

In [0]:
doc = uc1.withColumn("so_digitos", F.regexp_replace("cpf_cnpj", "[^0-9]", ""))
display(doc.groupBy(F.length("so_digitos").alias("qtd_digitos")).count().orderBy("qtd_digitos"))

por_doc = doc.groupBy("so_digitos").count()
display(por_doc.groupBy(F.col("count").alias("ucs_por_documento")).agg(F.count("*").alias("qtd_documentos"))
        .orderBy("ucs_por_documento"))

## 13. Checagens complementares

**Achados:**
- **Duplicatas:** depois de padronizar nome e documento, **0** ids com versões diferentes. A deduplicação não perde informação.
- **Mojibake:** só em `bairro`. A busca encontrou 388 linhas suspeitas; 32 são `Rio Âmbar`, que está **correto**. As outras são revertidas por uma função de codificação, mais um mapa manual para os poucos valores em que um caractere invisível se perdeu (`Nossa Senhora do Ã`, `Ãgua Fria`...).
- **Ligações antes de 1950:** 90, e só 1 com o mesmo dia e mês da instalação do primeiro medidor.

In [0]:
from pyspark.sql import functions as F

catalog = "v_energy"
def n_br(n): return f"{n:,}".replace(",", ".")
def sem_acento(col):
    c = F.translate(F.upper(F.trim(col)), "ÁÀÂÃÉÊÍÓÔÕÚÜÇ", "AAAAEEIOOOUUC")
    return F.regexp_replace(c, r"\s+", " ")
def converte(col, fmt):
    return F.expr(f"to_date(try_to_timestamp({col}, '{fmt}'))")

ucs = spark.table(f"{catalog}.bronze.unidades_consumidoras").filter(F.col("id_uc") != "id_uc")
cols_origem = [c for c in ucs.columns if c not in ("source_file", "ingestion_timestamp")]

# 1. Duplicatas depois de padronizar nome e documento
pad = (ucs.withColumn("nome_titular", sem_acento(F.col("nome_titular")))
          .withColumn("cpf_cnpj", F.regexp_replace("cpf_cnpj", "[^0-9]", ""))
          .select(cols_origem).distinct())
print("ids ainda com versões diferentes (nome e documento padronizados):",
      n_br(pad.groupBy("id_uc").count().filter("count > 1").count()))

# 2. Mojibake: em quais colunas e quais valores
for c in cols_origem:
    q = ucs.filter(F.col(c).rlike("Ã|Â")).count()
    if q: print(f"{c}: {n_br(q)} linhas com Ã ou Â")
display(ucs.filter(F.col("bairro").rlike("Ã|Â")).groupBy("bairro").count().orderBy(F.desc("count")))

# 3. Ligações antes de 1950 x instalação do 1º medidor
prim_med = (spark.table(f"{catalog}.silver.medidores")
            .groupBy("id_uc").agg(F.min("data_instalacao").alias("instalacao_1o_medidor")))
antigas = (ucs.dropDuplicates(["id_uc"]).withColumn("ligacao", converte("data_ligacao", "yyyy-MM-dd"))
           .filter("ligacao < DATE'1950-01-01'").join(prim_med, "id_uc", "left"))
print("ligações antes de 1950:", n_br(antigas.count()))
print("mesmo dia e mês da instalação:",
      n_br(antigas.filter("date_format(ligacao, 'MM-dd') = date_format(instalacao_1o_medidor, 'MM-dd')").count()))
display(antigas.select("id_uc", "data_ligacao", "instalacao_1o_medidor").limit(10))

## 14. Resumo e regras para a Silver

| Problema | Linhas | Regra | Log |
|---|---|---|---|
| Cabeçalho repetido | 1 | remover antes de tudo | U1 |
| Mesma UC com grafias diferentes de nome e documento | 1.260 | padronizar e `dropDuplicates(["id_uc"])` | U2 |
| UCs de teste (sem medidor, leitura ou fatura) | 200 | remover; total final 45.000 | U3 |
| `uf` fora de `PE` | 6.990 | `PE` | U4 |
| Grafias de `classe`, `municipio`, `status`; espaços em textos | várias | mapa para valores canônicos + `trim` | U5 |
| Mojibake em `bairro` | 356 (+32 `Rio Âmbar`, correto) | reverter a codificação + mapa manual | U6 |
| `cep` que não é CEP | 1.383 (7 formas) | nulo | U7 |
| `cpf_cnpj` com máscara | várias | só dígitos + `tipo_pessoa` | U8 |
| `data_ligacao` antes de 1950 | 90 | instalação do 1º medidor + flag `data_ligacao_estimada` | U9 |
| Dados pessoais | todas | manter na Silver; mascarar na Gold | U10 |

**Verificado e sem problema:** `id_uc` no padrão; classe × subgrupo conforme o glossário; alimentador × município coerente; cada UC real com exatamente 1 medidor ativo.